In [1]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

In [9]:
df = pd.read_csv('numeric_log_zscore_scaled.csv')

In [3]:
print(df.columns)

Index(['Date', 'SP500', 'SP500 Log Returns', 'SP500 30 Day Volatility',
       'SPX Put Call Ratio', 'SPX Put Volume', 'SPX Call Volume',
       'Total SPX Options Volume', 'VIX', 'DJIA', 'NASDAQ', '10Y_Treasury',
       'High_Yield_Bonds', 'RUSSELL', 'EMB_Yield', 'MSCI_World',
       'Consumer_Sentiment', 'USD_Index', 'Gold', 'Oil', 'DJIA_log_returns',
       'NASDAQ_log_returns', 'RUSSELL_log_returns', 'MSCI_World_log_returns',
       'USD_Index_log_returns', 'Gold_log_returns', 'Oil_log_returns',
       'SP500_put_log_change', 'SP500_call_log_change',
       'SP500_total_opts_log_change', 'Consumer_Sentiment_log_change',
       'SP500 30 Day Volatility_log', 'SPX Put Volume_log',
       'Total SPX Options Volume_log', 'VIX_log'],
      dtype='object')


In [10]:
cols_to_drop = [
    'SP500 30 Day Volatility',
    'SPX Put Volume',
    'Total SPX Options Volume',
    'VIX'
]

df= df.drop(columns=cols_to_drop)

In [11]:
# 設定目標變數與特徵變數
target_col = 'SP500 Log Returns'
feature_cols = [c for c in df.columns if c not in [target_col, 'Date']]

# 建立 lag 特徵（前 7 天）
max_lag = 7
for lag in range(1, max_lag + 1):
    lagged = df[feature_cols].shift(lag)
    lagged.columns = [f'{col}_lag{lag}' for col in feature_cols]
    df = pd.concat([df, lagged], axis=1)

# 移除前 7 筆（因為沒有完整的滯後資料）
df = df.dropna().reset_index(drop=True)

In [12]:
# 重新定義特徵矩陣 X 與目標 y
# 生成 lag 特徵後
lagged_feature_cols = [f'{col}_lag{lag}' for lag in range(1, max_lag+1) for col in feature_cols]

# 只保留 lagged feature 與目標
df_lagged = df[lagged_feature_cols + [target_col]].dropna().reset_index(drop=True)

# 定義 X, y
X = df_lagged[lagged_feature_cols]
y = df_lagged[target_col]


# 使用時間順序切分（避免資料外洩）
split_index = int(len(df) * 0.7)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# 建立並訓練模型
lr = LinearRegression()
lr.fit(X_train, y_train)

# 查看係數
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_
})

print(coef_df)
print("Intercept:", lr.intercept_)

                                Feature  Coefficient
0                            SP500_lag1    -8.204326
1               SPX Put Call Ratio_lag1    -1.136697
2                  SPX Call Volume_lag1     0.091365
3                             DJIA_lag1    -2.052023
4                           NASDAQ_lag1    22.020635
..                                  ...          ...
198  Consumer_Sentiment_log_change_lag7     0.032087
199    SP500 30 Day Volatility_log_lag7     0.119077
200             SPX Put Volume_log_lag7    -1.027389
201   Total SPX Options Volume_log_lag7     3.839202
202                        VIX_log_lag7     0.017169

[203 rows x 2 columns]
Intercept: -0.7847460023884084


In [13]:
y_pred = lr.predict(X_test)

print("R^2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R^2: -9.081285837926727
MSE: 8.132025477518656
MAE: 2.54460346116897
